In [1]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [4]:
import os

for root, dirs, files in os.walk("/content"):
    for file in files:
        if file.endswith(".csv"):
            print(os.path.join(root, file))

/content/sample_data/california_housing_test.csv
/content/sample_data/mnist_train_small.csv
/content/sample_data/california_housing_train.csv
/content/sample_data/mnist_test.csv


In [5]:
from google.colab import files

uploaded = files.upload()

Saving y_test.csv to y_test.csv
Saving X_test.csv to X_test.csv
Saving X_train.csv to X_train.csv
Saving X_val.csv to X_val.csv
Saving y_train.csv to y_train.csv
Saving y_val.csv to y_val.csv


In [6]:
import pandas as pd

X_train = pd.read_csv("/content/X_train.csv")
y_train = pd.read_csv("/content/y_train.csv").squeeze()

X_val = pd.read_csv("/content/X_val.csv")
y_val = pd.read_csv("/content/y_val.csv").squeeze()

X_test = pd.read_csv("/content/X_test.csv")
y_test = pd.read_csv("/content/y_test.csv").squeeze()

print("All datasets loaded successfully!")
print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape, y_test.shape)

All datasets loaded successfully!
(23714, 49) (23714,)
(5082, 49) (5082,)
(5082, 49) (5082,)


In [7]:
# Evaluation functions
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def evaluate_model(y_true, y_pred, name="Model"):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)

    print(f"{name} Performance:")
    print(f"  MAE : {mae:.2f}")
    print(f"  RMSE: {rmse:.2f}")
    print(f"  R²  : {r2:.3f}")
    print("-" * 30)


In [9]:
from sklearn.preprocessing import LabelEncoder

for col in ["station"]:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col].astype(str))
    X_val[col] = le.transform(X_val[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))

print("Categorical columns converted successfully!")

Categorical columns converted successfully!


In [10]:
# Train linear regression
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)

y_val_pred_lr = lin_reg.predict(X_val)

evaluate_model(y_val, y_val_pred_lr, name="Linear Regression (Validation)")

Linear Regression (Validation) Performance:
  MAE : 5.62
  RMSE: 9.96
  R²  : 0.980
------------------------------


In [11]:
# Train ridge regression
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)

y_val_pred_ridge = ridge.predict(X_val)

evaluate_model(y_val, y_val_pred_ridge, name="Ridge Regression (Validation)")


Ridge Regression (Validation) Performance:
  MAE : 5.67
  RMSE: 9.99
  R²  : 0.980
------------------------------


In [12]:
results = pd.DataFrame({
    "Model": ["Linear Regression", "Ridge Regression"],
    "MAE": [
        mean_absolute_error(y_val, y_val_pred_lr),
        mean_absolute_error(y_val, y_val_pred_ridge)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_val, y_val_pred_lr)),
        np.sqrt(mean_squared_error(y_val, y_val_pred_ridge))
    ],
    "R2": [
        r2_score(y_val, y_val_pred_lr),
        r2_score(y_val, y_val_pred_ridge)
    ]
})

results

,Model,MAE,RMSE,R2
0,Linear Regression,5.620863,9.958325,0.979634
1,Ridge Regression,5.672085,9.989290,0.979507


In [13]:
# Test evaluation for ridge regression
y_test_pred = ridge.predict(X_test)

evaluate_model(y_test, y_test_pred, name="Ridge Regression (Test)")

Ridge Regression (Test) Performance:
  MAE : 6.12
  RMSE: 10.28
  R²  : 0.988
------------------------------


In [14]:
# Test evaluation for linear regression
y_test_pred = lin_reg.predict(X_test)

evaluate_model(y_test, y_test_pred, name="Linear Regression (Test)")

Linear Regression (Test) Performance:
  MAE : 6.06
  RMSE: 10.25
  R²  : 0.988
------------------------------


In [16]:
from pathlib import Path

OUTPUT_DIR = Path("/content/beijing_data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

predictions = pd.DataFrame({
    "y_true": y_test,
    "y_pred_ridge": y_test_pred
})

predictions.to_csv(OUTPUT_DIR / "baseline_predictions.csv", index=False)

print("Baseline predictions saved successfully!")
print(predictions.head())

Baseline predictions saved successfully!
   y_true  y_pred_ridge
0    59.0     75.582563
1    60.0     54.124903
2    70.0     63.458379
3    77.0     71.527295
4    71.0     73.173729


In [17]:
from google.colab import files
files.download("/content/beijing_data/processed/baseline_predictions.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>